# TripoSR · Kaggle 双 T4 常驻 Worker

这份 Notebook 专门用于 Kaggle：Notebook 内核可以保持 **Python 3.12**，TripoSR 始终通过 `uv` 创建的 **Python 3.10 `.venv`** 运行。

设计目标：

- TripoSR **每张 T4 常驻一份模型**，不再每个任务重新执行 `run.py` / 重载 1.68 GB 权重。
- `rembg[gpu]` + `onnxruntime-gpu`，并把两个 rembg Session 分别固定到 GPU0 / GPU1。
- `torchmcubes` 直接安装 `kaggle-build` Release 的 `sm_75` wheel，**不现场 CMake/NVCC 编译**。
- 两个 GPU 进程各自 long-poll `triposr` 队列，天然双卡并行。
- 输入只在内存中处理，结果 GLB/OBJ 经过 AES-GCM 加密后上传回 Hub。
- 模型下载和初始化只发生在 Worker 启动阶段；后续单张任务只承担 rembg + TripoSR + mesh + 上传。
- Hub 地址和鉴权/AES 密钥支持环境变量或 Kaggle Secrets；未配置时回退到 Hub 的默认值。

Kaggle 要求：**GPU T4 x2 + Internet**。


## Cell 1 · 安装 Python 3.10 TripoSR Runtime

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ["UV_LINK_MODE"] = "copy"
os.environ.pop("UV_SYSTEM_PYTHON", None)
os.environ["IPYTHONDIR"] = "/kaggle/working/.ipython"

ROOT = Path("/kaggle/working/TripoSR")
UV = "/usr/local/bin/uv"
PY = ROOT / ".venv/bin/python"
REPO = "https://github.com/VAST-AI-Research/TripoSR.git"
WHEEL_URL = (
    "https://github.com/xiaoqianran/kaggle-build/releases/download/"
    "triposr-py310-torch2.7.1-cu128-sm75/"
    "torchmcubes-0.1.0-cp310-cp310-linux_x86_64.whl"
)

def run(cmd, cwd=None):
    print("\n>>>", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True, cwd=str(cwd) if cwd else None)

if ROOT.exists():
    shutil.rmtree(ROOT)

run(["git", "clone", "--depth", "1", REPO, ROOT])
run([UV, "venv", "--no-managed-python", "--python", "/usr/bin/python3.10", ROOT / ".venv"])
run([
    UV, "pip", "install", "--python", PY,
    "torch==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu128",
])

req = ROOT / "requirements.txt"
req_fixed = ROOT / "requirements.kaggle.txt"
req_fixed.write_text(
    "\n".join(line for line in req.read_text().splitlines() if "torchmcubes" not in line.lower()) + "\n"
)
run([UV, "pip", "install", "--python", PY, "-r", req_fixed])
run([UV, "pip", "install", "--python", PY, "rembg[gpu]", "cryptography", "requests"])
run([UV, "pip", "install", "--python", PY, WHEEL_URL])

check = r"""
import torch, torchmcubes, onnxruntime as ort
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}:", torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
print("ORT:", ort.__version__)
print("ORT providers:", ort.get_available_providers())
assert torch.cuda.is_available()
assert "CUDAExecutionProvider" in ort.get_available_providers()
x=torch.randn(16,16,16,device="cuda:0")
v,f=torchmcubes.marching_cubes(x,0.0)
print("torchmcubes CUDA OK:", v.shape, f.shape)
"""
run([PY, "-c", check])
print("\n✅ Python 3.10 TripoSR runtime ready:", PY)

## Cell 2 · 写入常驻双 GPU Worker

Notebook 的 Python 3.12 **不直接 import TripoSR**。这一格只把 Worker 写成 `.py` 文件，后面始终使用 `.venv/bin/python` 启动。

In [ ]:
from pathlib import Path

WORKER = Path("/kaggle/working/TripoSR/kaggle_worker.py")
WORKER_SOURCE = 'from __future__ import annotations\n\nimport hashlib\nimport io\nimport multiprocessing as mp\nimport os\nimport signal\nimport socket\nimport threading\nimport time\nimport traceback\nimport uuid\nfrom typing import Any\nfrom urllib.parse import urljoin\n\nimport numpy as np\nimport rembg\nimport requests\nimport torch\nfrom cryptography.hazmat.primitives.ciphers.aead import AESGCM\nfrom PIL import Image\nfrom tsr.system import TSR\nfrom tsr.utils import remove_background, resize_foreground\n\nBASE_URL = os.environ["BASE_URL"].rstrip("/") + "/"\nTOKEN = os.environ["KAGGLE_HUB_TOKEN"]\nMODEL_ID = os.getenv("TRIPOSR_MODEL_ID", "stabilityai/TripoSR")\nREMBG_MODEL = os.getenv("TRIPOSR_REMBG_MODEL", "u2net")\nPOLL_TIMEOUT = 35\nREQUEST_TIMEOUT = 120\nHEARTBEAT_SECONDS = 10\n\n\ndef encrypt_blob(data: bytes) -> bytes:\n    key = hashlib.sha256(TOKEN.encode()).digest()\n    nonce = os.urandom(12)\n    return nonce + AESGCM(key).encrypt(nonce, data, None)\n\n\ndef api_url(path: str) -> str:\n    return urljoin(BASE_URL, path.lstrip("/"))\n\n\ndef auth_headers() -> dict[str, str]:\n    return {"Authorization": f"Bearer {TOKEN}"}\n\n\ndef prefetch_model() -> None:\n    from huggingface_hub import snapshot_download\n\n    print(f"[prefetch] {MODEL_ID}", flush=True)\n    snapshot_download(repo_id=MODEL_ID, allow_patterns=["config.yaml", "model.ckpt"])\n    print("[prefetch] TripoSR assets ready", flush=True)\n\n\ndef build_rembg_session(gpu: int):\n    providers = [\n        ("CUDAExecutionProvider", {"device_id": gpu}),\n        "CPUExecutionProvider",\n    ]\n    session = rembg.new_session(REMBG_MODEL, providers=providers)\n    active = session.inner_session.get_providers()\n    options = session.inner_session.get_provider_options()\n    if "CUDAExecutionProvider" not in active:\n        raise RuntimeError(f"rembg CUDA provider unavailable on GPU{gpu}: {active}")\n    print(\n        f"[GPU{gpu}] rembg={REMBG_MODEL} providers={active} options={options.get(\'CUDAExecutionProvider\', {})}",\n        flush=True,\n    )\n    return session\n\n\ndef prepare_image(raw: bytes, session, remove_bg: bool, foreground_ratio: float) -> Image.Image:\n    image = Image.open(io.BytesIO(raw))\n    if not remove_bg:\n        return image.convert("RGB")\n\n    image = remove_background(image, session)\n    image = resize_foreground(image, foreground_ratio)\n    arr = np.asarray(image).astype(np.float32) / 255.0\n    arr = arr[:, :, :3] * arr[:, :, 3:4] + (1.0 - arr[:, :, 3:4]) * 0.5\n    return Image.fromarray((arr * 255.0).astype(np.uint8))\n\n\ndef export_mesh(mesh, output_format: str) -> bytes:\n    data = mesh.export(file_type=output_format)\n    if isinstance(data, str):\n        return data.encode("utf-8")\n    return bytes(data)\n\n\ndef heartbeat_loop(worker_id: str, gpu: int, stop: threading.Event) -> None:\n    session = requests.Session()\n    session.headers.update(auth_headers())\n    while not stop.wait(HEARTBEAT_SECONDS):\n        try:\n            session.post(\n                api_url("/worker/heartbeat"),\n                json={\n                    "worker_id": worker_id,\n                    "local_queue": 0,\n                    "upload_queue": 0,\n                    "meta": {"gpu_index": gpu, "persistent": True},\n                },\n                timeout=15,\n            ).raise_for_status()\n        except Exception as exc:\n            print(f"[GPU{gpu}] heartbeat: {type(exc).__name__}: {exc}", flush=True)\n\n\ndef report_failure(session: requests.Session, task_id: int, exc: BaseException, gpu: int) -> None:\n    message = f"{type(exc).__name__}: {exc}"\n    print(f"[GPU{gpu}] FAIL #{task_id}: {message}", flush=True)\n    try:\n        session.post(\n            api_url("/task/fail"),\n            json={"id": task_id, "error": message[:1900], "requeue": True},\n            timeout=20,\n        ).raise_for_status()\n    except Exception as report_exc:\n        print(f"[GPU{gpu}] fail-report error: {report_exc}", flush=True)\n\n\ndef gpu_worker(gpu: int, run_id: str) -> None:\n    torch.cuda.set_device(gpu)\n    device = f"cuda:{gpu}"\n    gpu_name = torch.cuda.get_device_name(gpu)\n    worker_id = f"triposr-{run_id}-g{gpu}"\n\n    print(f"[GPU{gpu}] loading TripoSR on {gpu_name} ...", flush=True)\n    started = time.perf_counter()\n    model = TSR.from_pretrained(MODEL_ID, config_name="config.yaml", weight_name="model.ckpt")\n    model.renderer.set_chunk_size(8192)\n    model.to(device)\n    model.eval()\n    rembg_session = build_rembg_session(gpu)\n    print(f"[GPU{gpu}] READY in {time.perf_counter()-started:.2f}s", flush=True)\n\n    session = requests.Session()\n    session.headers.update(auth_headers())\n    session.post(\n        api_url("/worker/register"),\n        json={\n            "worker_id": worker_id,\n            "model": "triposr",\n            "gpus": [gpu_name],\n            "runtime": "triposr-persistent-py310",\n            "concurrency": 1,\n            "meta": {\n                "gpu_index": gpu,\n                "torch": torch.__version__,\n                "torch_cuda": torch.version.cuda,\n                "rembg_model": REMBG_MODEL,\n                "rembg_providers": rembg_session.inner_session.get_providers(),\n                "persistent": True,\n            },\n        },\n        timeout=30,\n    ).raise_for_status()\n\n    stop = threading.Event()\n    heartbeat = threading.Thread(target=heartbeat_loop, args=(worker_id, gpu, stop), daemon=True)\n    heartbeat.start()\n\n    def shutdown(*_args):\n        stop.set()\n        raise KeyboardInterrupt\n\n    signal.signal(signal.SIGTERM, shutdown)\n\n    try:\n        while True:\n            task: dict[str, Any] | None = None\n            try:\n                response = session.get(\n                    api_url("/task/next"),\n                    params={"model": "triposr", "worker_id": worker_id},\n                    timeout=POLL_TIMEOUT,\n                )\n                if response.status_code == 204:\n                    continue\n                response.raise_for_status()\n                task = response.json()\n                task_id = int(task["id"])\n                print(\n                    f"[GPU{gpu}] ↓ #{task_id} {task.get(\'source_label\',\'input\')} "\n                    f"res={task.get(\'mc_resolution\',256)} fmt={task.get(\'output_format\',\'glb\')}",\n                    flush=True,\n                )\n\n                t0 = time.perf_counter()\n                input_response = session.get(api_url(task["input_url"]), timeout=60)\n                input_response.raise_for_status()\n                t_download = time.perf_counter()\n\n                image = prepare_image(\n                    input_response.content,\n                    rembg_session,\n                    bool(task.get("remove_background", True)),\n                    float(task.get("foreground_ratio", 0.85)),\n                )\n                t_pre = time.perf_counter()\n\n                model.renderer.set_chunk_size(int(task.get("chunk_size", 8192)))\n                with torch.inference_mode():\n                    scene_codes = model([image], device=device)\n                t_model = time.perf_counter()\n\n                with torch.inference_mode():\n                    meshes = model.extract_mesh(\n                        scene_codes,\n                        True,\n                        resolution=int(task.get("mc_resolution", 256)),\n                    )\n                t_mesh = time.perf_counter()\n\n                mesh = meshes[0]\n                output_format = str(task.get("output_format", "glb")).lower()\n                artifact = export_mesh(mesh, output_format)\n                encrypted = encrypt_blob(artifact)\n                t_export = time.perf_counter()\n\n                elapsed = t_export - t0\n                upload = session.post(\n                    api_url("/upload/artifact"),\n                    data={\n                        "id": str(task_id),\n                        "model": "triposr",\n                        "worker_id": worker_id,\n                        "gpu": str(gpu),\n                        "seconds": f"{elapsed:.3f}",\n                        "output_format": output_format,\n                        "vertices": str(len(mesh.vertices)),\n                        "faces": str(len(mesh.faces)),\n                    },\n                    files={\n                        "file": (\n                            f"{task_id}.{output_format}.bin",\n                            encrypted,\n                            "application/octet-stream",\n                        )\n                    },\n                    timeout=REQUEST_TIMEOUT,\n                )\n                upload.raise_for_status()\n                t_up = time.perf_counter()\n\n                print(\n                    f"[GPU{gpu}] ✓ #{task_id} total={elapsed:.2f}s "\n                    f"download={t_download-t0:.2f}s rembg={t_pre-t_download:.2f}s "\n                    f"model={t_model-t_pre:.2f}s mesh={t_mesh-t_model:.2f}s "\n                    f"export={t_export-t_mesh:.2f}s upload={t_up-t_export:.2f}s "\n                    f"v={len(mesh.vertices)} f={len(mesh.faces)}",\n                    flush=True,\n                )\n\n                del image, scene_codes, meshes, mesh, artifact, encrypted\n                torch.cuda.empty_cache()\n\n            except KeyboardInterrupt:\n                raise\n            except Exception as exc:\n                if task is not None and "id" in task:\n                    report_failure(session, int(task["id"]), exc, gpu)\n                else:\n                    print(f"[GPU{gpu}] poll error: {type(exc).__name__}: {exc}", flush=True)\n                    time.sleep(2)\n                traceback.print_exc()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        stop.set()\n        print(f"[GPU{gpu}] stopped", flush=True)\n\n\ndef main() -> None:\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is not available")\n    gpu_count = torch.cuda.device_count()\n    if gpu_count < 1:\n        raise RuntimeError("No CUDA GPU found")\n\n    wanted = int(os.getenv("TRIPOSR_GPU_COUNT", str(gpu_count)))\n    gpu_count = min(gpu_count, max(1, wanted))\n    run_id = f"{socket.gethostname()[:8]}-{uuid.uuid4().hex[:6]}"\n\n    print(\n        f"TripoSR persistent worker | GPUs={gpu_count} | rembg={REMBG_MODEL} | base={BASE_URL}",\n        flush=True,\n    )\n    prefetch_model()\n\n    ctx = mp.get_context("spawn")\n    processes = [\n        ctx.Process(target=gpu_worker, args=(gpu, run_id), name=f"triposr-gpu{gpu}")\n        for gpu in range(gpu_count)\n    ]\n    for process in processes:\n        process.start()\n\n    try:\n        for process in processes:\n            process.join()\n    except KeyboardInterrupt:\n        print("Stopping workers ...", flush=True)\n        for process in processes:\n            if process.is_alive():\n                process.terminate()\n        for process in processes:\n            process.join(timeout=10)\n\n\nif __name__ == "__main__":\n    mp.freeze_support()\n    main()\n'
WORKER.write_text(WORKER_SOURCE)
print("✅ Worker written:", WORKER)
print("Lines:", len(WORKER_SOURCE.splitlines()))

## Cell 3 · 启动双 T4 常驻 Worker

配置优先级为：Cell 外部环境变量 → Kaggle Secrets → Hub 默认值。推荐在 Kaggle Secrets 中配置 `BASE_URL` 和 `KAGGLE_HUB_TOKEN`：

- Hub Secret：`BASE_URL`
- Bearer / AES-GCM Secret：`KAGGLE_HUB_TOKEN`
- rembg：`u2net` + CUDAExecutionProvider

如需临时切换，也可以在启动本 Cell 前设置同名环境变量。Token 不会打印到 Notebook 输出。


In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/TripoSR")
PY = ROOT / ".venv/bin/python"
WORKER = ROOT / "kaggle_worker.py"
LOG = Path("/kaggle/working/triposr-worker.log")
PID_FILE = Path("/kaggle/working/triposr-worker.pid")

DEFAULT_BASE_URL = "https://ranran-sana.202820.xyz"
DEFAULT_TOKEN = "wangran"

def kaggle_secret(name: str, default: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = ""
    return (value or "").strip() or default

def setting(name: str, default: str) -> str:
    return os.getenv(name, "").strip() or kaggle_secret(name, default)

BASE_URL = setting("BASE_URL", DEFAULT_BASE_URL).rstrip("/")
TOKEN = (
    os.getenv("KAGGLE_HUB_TOKEN", "").strip()
    or os.getenv("PASSWORD", "").strip()
    or kaggle_secret("KAGGLE_HUB_TOKEN", DEFAULT_TOKEN)
)

REMBG_MODEL = "u2net"
GPU_COUNT = 2

if PID_FILE.exists():
    try:
        old_pid = int(PID_FILE.read_text().strip())
        os.kill(old_pid, 0)
        raise RuntimeError(f"Worker 已在运行 PID={old_pid}；先执行停止 Cell")
    except ProcessLookupError:
        PID_FILE.unlink(missing_ok=True)

LOG.write_text("")
env = os.environ.copy()
env.update({
    "BASE_URL": BASE_URL,
    "KAGGLE_HUB_TOKEN": TOKEN,
    "TRIPOSR_REMBG_MODEL": REMBG_MODEL,
    "TRIPOSR_GPU_COUNT": str(GPU_COUNT),
    "HF_HOME": "/kaggle/working/hf-cache",
    "U2NET_HOME": "/kaggle/working/.u2net",
    "PYTHONUNBUFFERED": "1",
})

log_handle = LOG.open("ab", buffering=0)
proc = subprocess.Popen(
    [str(PY), str(WORKER)],
    cwd=str(ROOT),
    env=env,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
PID_FILE.write_text(str(proc.pid))

print("✅ TripoSR persistent worker started")
print("PID:", proc.pid)
print("Hub:", BASE_URL)
print("Token: configured (value hidden)")
print("Log:", LOG)
print("前几十秒会下载/加载模型；之后两个 GPU 常驻，不再按任务重新初始化。")


## Cell 4 · 查看 Worker 状态 / 最近日志

In [ ]:
import os
import subprocess
from pathlib import Path

PID_FILE = Path("/kaggle/working/triposr-worker.pid")
LOG = Path("/kaggle/working/triposr-worker.log")

if PID_FILE.exists():
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, 0)
        print("Worker: RUNNING | PID", pid)
    except ProcessLookupError:
        print("Worker: EXITED | PID", pid)
else:
    print("Worker: NOT STARTED")

print("\n=== GPU ===")
subprocess.run(["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total", "--format=csv"])
print("\n=== Last 100 log lines ===")
if LOG.exists():
    subprocess.run(["tail", "-n", "100", str(LOG)])

## Cell 5 · 停止 Worker

In [ ]:
import os
import signal
import time
from pathlib import Path

PID_FILE = Path("/kaggle/working/triposr-worker.pid")
if not PID_FILE.exists():
    print("没有运行中的 Worker")
else:
    pid = int(PID_FILE.read_text().strip())
    try:
        os.killpg(pid, signal.SIGTERM)
        print("Stopping process group:", pid)
        time.sleep(2)
    except ProcessLookupError:
        pass
    PID_FILE.unlink(missing_ok=True)
    print("✅ stopped")

## 运行方式

1. 本地启动 Hub，并让 Cloudflare Tunnel 指向 `BASE_URL` 对应的地址。
2. 推荐在 Kaggle Secrets 中添加 `BASE_URL` 和 `KAGGLE_HUB_TOKEN`，再依次执行 Cell 1 → 2 → 3。未添加时会使用 Hub 默认值。
3. 等 Cell 4 日志同时出现 `GPU0 READY` 与 `GPU1 READY`。
4. 在本地 UI 上传图片或在生成图上点击“转 3D”。
5. 两张 T4 会分别领取任务；模型与 rembg Session 始终常驻。

Notebook 的配置来源：

```text
BASE_URL：环境变量或 Kaggle Secret `BASE_URL`，否则使用 Hub 默认地址
KAGGLE_HUB_TOKEN：环境变量或 Kaggle Secret `KAGGLE_HUB_TOKEN`，否则使用 Hub 默认密钥
```

正常任务日志类似：

```text
[GPU0] ↓ #41 chair.png res=256 fmt=glb
[GPU0] ✓ #41 total=5.12s download=0.08s rembg=0.63s model=1.91s mesh=2.28s export=0.22s ...
```
